# Monte Carlo Option Pricing and Sensitivity Validation

## Research question
How accurately and efficiently do Monte Carlo estimators value European and discretely monitored arithmetic Asian options under a common risk-neutral GBM model, and how reliably do finite-difference estimates reproduce analytical European sensitivities?

**Status:** executable research implementation; not a production trading or enterprise risk system. Local execution and review are required before publishing results. No market calibration or empirical trading-performance claim is made.

### Contents
1. Scope, contracts and conventions
2. Reproducible configuration and exports
3. Analytical BSM benchmark and Greeks
4. Exact simulation and estimator design
5. Deterministic and statistical validation
6. European pricing and parameter-grid checks
7. Repeated-run convergence, interval coverage and efficiency
8. Finite-difference sensitivities and bump diagnostics
9. Arithmetic Asian extension and geometric control
10. Interpretation, limitations and reproducibility record

Run all cells in order in a fresh Python kernel. No network connection, API key, or external price CSV is required. Synthetic model inputs are deliberate: an analytical benchmark lets us isolate implementation error from market-model error.

## 1. Contracts, assumptions and conventions

Under the pricing measure, $dS_t=(r-q)S_tdt+\sigma S_tdW_t$. Rates $r,q$ are continuously compounded annual rates; volatility is annualised; maturity $T$ is in years. Prices are per unit underlying, with no multiplier. The model has constant inputs, no transaction costs, no default and no stochastic rates.

European payoffs are $(S_T-K)^+$ and $(K-S_T)^+$. Values are $e^{-rT}\mathbb E^Q[g(S_T)]$. Historical return forecasting is not risk-neutral pricing.

Asian contracts use an equally weighted arithmetic average at $t_j=jT/m$, $j=1,\ldots,m$. **The initial spot is excluded.** Changing $m$ changes the contract, not merely a solver tolerance. All comparisons within an Asian experiment use the same schedule.

Greeks: delta per spot unit, gamma per spot-unit squared, vega per 1.00 change in volatility, rho per 1.00 change in rate, theta per calendar year elapsed ($-\partial V/\partial T$). Vega/rho per percentage point are the raw values divided by 100; annual theta divided by 365 is a reporting convention, not a business-day model.

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict, replace
from time import perf_counter
from datetime import datetime, timezone
import sys, platform, json, hashlib, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, t as student_t, binomtest
from IPython.display import display

# Set PROJECT_ROOT explicitly if launching from another directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS = PROJECT_ROOT / 'results'
FIGURES, TABLES = RESULTS / 'figures', RESULTS / 'tables'
for folder in (FIGURES, TABLES):
    folder.mkdir(parents=True, exist_ok=True)

MODE = 'quick'  # 'full' increases precision and runtime, not model realism.
if MODE not in {'quick', 'full'}:
    raise ValueError('MODE must be quick or full')
CFG = dict(seed=20261117, n=20000, pilot=4000, repeats=24,
           convergence_n=[1000, 4000, 16000], greek_n=40000,
           asian_n=12000, asian_pilot=3000, monitoring=52)
if MODE == 'full':
    CFG.update(n=200000, pilot=20000, repeats=150,
               convergence_n=[2000, 8000, 32000, 128000],
               greek_n=400000, asian_n=100000, asian_pilot=15000)

def rng_for(*keys):
    return np.random.default_rng(np.random.SeedSequence([CFG['seed'], *map(int, keys)]))

def save_table(df, name):
    df.to_csv(TABLES / f'{name}.csv', index=False)
    return df

def finish_figure(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / f'{name}.png', dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)

plt.rcParams.update({'figure.figsize': (9, 5), 'axes.grid': True,
                     'grid.alpha': .22, 'font.size': 10})
pd.set_option('display.max_columns', 20)

## 2. Source provenance and rebuilding decisions

The four supplied notebooks are source exercises, not validated reference implementations. The short BSM upload contains analytical pricing and a CSV-dependent illustration; analytical Greeks and the integrated simulation framework below are newly implemented. The rebuild removes the external CSV dependency and ambiguous hourly units.

- Antithetics: average **payoffs**, not terminal stock prices. Treat each pair average as one independent observation.
- Control variates: divide covariance by variance, use the matching payoff, and estimate the coefficient from an independent pilot sample. Pilot cost is included in timings.
- Exact GBM replaces Euler simulation for the baseline. No sample moment matching is applied; uncertainty calculations use independent draws or independent pairs.
- No comparison between two noisy prices is labelled an error without a benchmark.
- Recompute all results; do not import old charts as evidence for this implementation.

In [ ]:
@dataclass(frozen=True)
class Contract:
    spot: float = 100.0
    strike: float = 100.0
    rate: float = 0.05
    dividend: float = 0.0
    vol: float = 0.20
    maturity: float = 1.0
    kind: str = 'call'

    def __post_init__(self):
        values = [self.spot, self.strike, self.rate, self.dividend, self.vol, self.maturity]
        if not np.all(np.isfinite(values)):
            raise ValueError('All numerical inputs must be finite')
        if self.spot <= 0 or self.strike <= 0 or self.vol < 0 or self.maturity < 0:
            raise ValueError('Spot/strike must be positive; vol/maturity nonnegative')
        if self.kind not in {'call', 'put'}:
            raise ValueError('kind must be call or put')

BASE = Contract()

def payoff(x, c):
    sign = 1 if c.kind == 'call' else -1
    return np.maximum(sign * (np.asarray(x) - c.strike), 0.0)

def bsm_price(c):
    if c.maturity == 0:
        return float(payoff(c.spot, c))
    ds = c.spot * np.exp(-c.dividend*c.maturity)
    dk = c.strike * np.exp(-c.rate*c.maturity)
    if c.vol == 0:
        return float(max(ds-dk, 0) if c.kind == 'call' else max(dk-ds, 0))
    d1 = (np.log(c.spot/c.strike)+(c.rate-c.dividend+.5*c.vol**2)*c.maturity)/(c.vol*np.sqrt(c.maturity))
    d2 = d1-c.vol*np.sqrt(c.maturity)
    return float(ds*norm.cdf(d1)-dk*norm.cdf(d2) if c.kind == 'call'
                 else dk*norm.cdf(-d2)-ds*norm.cdf(-d1))

def bsm_greeks(c):
    if c.vol <= 0 or c.maturity <= 0:
        raise ValueError('Smooth Greeks require strictly positive vol and maturity')
    S,K,r,q,v,T = c.spot,c.strike,c.rate,c.dividend,c.vol,c.maturity
    d1 = (np.log(S/K)+(r-q+.5*v*v)*T)/(v*np.sqrt(T))
    d2 = d1-v*np.sqrt(T)
    dq, dr, density = np.exp(-q*T), np.exp(-r*T), norm.pdf(d1)
    gamma = dq*density/(S*v*np.sqrt(T))
    vega = S*dq*density*np.sqrt(T)
    common = -S*dq*density*v/(2*np.sqrt(T))
    if c.kind == 'call':
        delta = dq*norm.cdf(d1)
        theta = common-r*K*dr*norm.cdf(d2)+q*S*dq*norm.cdf(d1)
        rho = K*T*dr*norm.cdf(d2)
    else:
        delta = dq*(norm.cdf(d1)-1)
        theta = common+r*K*dr*norm.cdf(-d2)-q*S*dq*norm.cdf(-d1)
        rho = -K*T*dr*norm.cdf(-d2)
    return dict(delta=delta, gamma=gamma, vega=vega, theta=theta, rho=rho)

display(pd.DataFrame([{'kind': k, 'price': bsm_price(replace(BASE,kind=k)),
                       **bsm_greeks(replace(BASE,kind=k))} for k in ('call','put')]))

## 3. Exact simulation and uncertainty

For European options only terminal values are needed:
$$S_T=S_0\exp[(r-q-\sigma^2/2)T+\sigma\sqrt{T}Z],\quad Z\sim N(0,1).$$
This avoids time-discretisation error under GBM. A finite number of Asian monitoring dates is simulated with exact GBM transitions.

For independent discounted observations $Y_i$, report $\bar Y$, $s_Y/\sqrt n$ and an approximate 95% Student interval. For antithetics the observations are paired payoff averages, not individual legs. Coverage is asymptotic, not guaranteed for skewed, sparse-payoff samples. A zero estimated SE can be misleading when no rare payoff was observed.

The European control is $X=e^{-rT}S_T$, with known mean $S_0e^{-qT}$. Use $Y^*=Y-\hat\beta(X-\mu_X)$ and $\hat\beta=\widehat{\mathrm{Cov}}(Y,X)/\widehat{\mathrm{Var}}(X)$. A separate pilot makes the production observations independent conditional on the fitted coefficient. Confidence intervals are conditional on that pilot; repeated-run experiments refit it each time.

In [ ]:
def terminal(c, z):
    return c.spot*np.exp((c.rate-c.dividend-.5*c.vol**2)*c.maturity
                        +c.vol*np.sqrt(c.maturity)*np.asarray(z))

def discounted_payoff(c, z):
    return np.exp(-c.rate*c.maturity)*payoff(terminal(c,z),c)

def summarize(samples, confidence=.95):
    y = np.asarray(samples, dtype=float)
    if y.ndim != 1 or len(y) < 2 or not np.all(np.isfinite(y)):
        raise ValueError('Need at least two finite independent observations')
    n = len(y)
    mean = float(y.mean())
    se = float(y.std(ddof=1)/np.sqrt(n))
    margin = student_t.ppf((1+confidence)/2,n-1)*se
    return dict(estimate=mean, se=se, ci_low=mean-margin, ci_high=mean+margin,
                n_independent=n, zero_se=(se == 0))

def fit_beta(y, x):
    var = np.var(x, ddof=1)
    return float(np.cov(y,x,ddof=1)[0,1]/var) if var > 1e-24 else 0.0

def european_estimate(c, method, budget, rng, pilot=4000):
    if method not in {'plain','antithetic','control'}:
        raise ValueError('Unknown method')
    if not isinstance(budget,(int,np.integer)) or budget < 4:
        raise ValueError('budget must be an integer >=4')
    if method == 'control' and pilot < 2:
        raise ValueError('pilot must be >=2')
    start = perf_counter()
    beta, extra = np.nan, 0
    if method == 'antithetic':
        z = rng.standard_normal(budget//2)
        samples = .5*(discounted_payoff(c,z)+discounted_payoff(c,-z))
        production = 2*len(z)
    else:
        if method == 'control':
            zp = rng.standard_normal(pilot)
            xp = np.exp(-c.rate*c.maturity)*terminal(c,zp)
            beta = fit_beta(discounted_payoff(c,zp),xp)
            extra = pilot
        z = rng.standard_normal(budget)
        samples = discounted_payoff(c,z)
        if method == 'control':
            x = np.exp(-c.rate*c.maturity)*terminal(c,z)
            samples = samples-beta*(x-c.spot*np.exp(-c.dividend*c.maturity))
        production = budget
    result = summarize(samples)
    result.update(method=method, beta=beta, production_evals=production,
                  pilot_evals=extra, total_evals=production+extra,
                  seconds=perf_counter()-start)
    return result

## 4. Validation gates

Deterministic identities and finite-difference checks are hard gates. Statistical diagnostics are recorded separately: a valid Monte Carlo run can miss a nominal confidence interval, so an individual miss is not automatically an implementation failure. Passing these checks is not independent model certification.

In [ ]:
checks=[]
def check(name, condition):
    checks.append({'check':name,'passed':bool(condition)})
    if not condition:
        raise AssertionError(name)

for K in (70.,100.,140.):
    for T in (.1,1.,3.):
        c=replace(BASE,strike=K,maturity=T,dividend=.02)
        put=replace(c,kind='put')
        parity=c.spot*np.exp(-c.dividend*T)-K*np.exp(-c.rate*T)
        check(f'put_call_parity_{K}_{T}', np.isclose(bsm_price(c)-bsm_price(put),parity,atol=1e-10))
        check(f'call_bounds_{K}_{T}', max(parity,0)-1e-10 <= bsm_price(c) <= c.spot*np.exp(-c.dividend*T)+1e-10)
        check(f'put_bounds_{K}_{T}', max(-parity,0)-1e-10 <= bsm_price(put) <= K*np.exp(-c.rate*T)+1e-10)
check('known_call', np.isclose(bsm_price(BASE),10.450583572185565,atol=1e-10))
check('expiry', bsm_price(replace(BASE,maturity=0,spot=110))==10)
check('zero_vol', np.isclose(bsm_price(replace(BASE,vol=0)),100-100*np.exp(-.05)))
check('positive_terminal_values',np.all(terminal(BASE,rng_for(1).normal(size=1000))>0))
for kind in ('call','put'):
    c=replace(BASE,kind=kind,dividend=.02)
    a=bsm_greeks(c)
    for name,field,h,sgn in [('delta','spot',.01,1),('vega','vol',1e-5,1),
                              ('rho','rate',1e-5,1),('theta','maturity',1e-5,-1)]:
        v=getattr(c,field)
        fd=sgn*(bsm_price(replace(c,**{field:v+h}))-bsm_price(replace(c,**{field:v-h})))/(2*h)
        check(f'{kind}_{name}',np.isclose(fd,a[name],rtol=2e-5,atol=1e-6))
    h=.01
    gamma=(bsm_price(replace(c,spot=c.spot+h))-2*bsm_price(c)+bsm_price(replace(c,spot=c.spot-h)))/h**2
    check(f'{kind}_gamma',np.isclose(gamma,a['gamma'],rtol=2e-5))
a=european_estimate(BASE,'control',4000,rng_for(2),1000)
b=european_estimate(BASE,'control',4000,rng_for(2),1000)
check('seed_reproducibility',a['estimate']==b['estimate'])
display(save_table(pd.DataFrame(checks),'validation_checks'))

z=rng_for(3).normal(size=CFG['n'])
x=np.exp(-BASE.rate*BASE.maturity)*terminal(BASE,z)
mart=summarize(x)
mart['known_mean']=BASE.spot*np.exp(-BASE.dividend*BASE.maturity)
mart['z_score']=(mart['estimate']-mart['known_mean'])/mart['se']
display(save_table(pd.DataFrame([mart]),'discounted_stock_diagnostic'))

## 5. European benchmark and parameter grid

The primary comparison uses equal production payoff counts. Antithetics use half as many independent normal draws but two payoff evaluations per pair. Controls use an additional pilot; total evaluations and timing include that overhead. This is not an equal-total-cost experiment. Runtime depends on hardware, allocation and system load; repeated-run variance × mean runtime below is an additional, noisy efficiency diagnostic.

In [ ]:
METHODS=('plain','antithetic','control')
rows=[]
for ki,kind in enumerate(('call','put')):
    c=replace(BASE,kind=kind)
    exact=bsm_price(c)
    for mi,method in enumerate(METHODS):
        row=european_estimate(c,method,CFG['n'],rng_for(10,ki,mi),CFG['pilot'])
        row.update(kind=kind, benchmark=exact, error=row['estimate']-exact,
                   covered=row['ci_low']<=exact<=row['ci_high'])
        rows.append(row)
european=save_table(pd.DataFrame(rows),'european_comparison')
display(european)

rows=[]
for ki,kind in enumerate(('call','put')):
    for K in (80.,100.,120.):
        for T in (.25,1.,2.):
            c=replace(BASE,kind=kind,strike=K,maturity=T)
            row=european_estimate(c,'control',CFG['n'],rng_for(11,ki,int(K),int(T*100)),CFG['pilot'])
            exact=bsm_price(c)
            row.update(kind=kind,strike=K,maturity=T,benchmark=exact,error=row['estimate']-exact,
                       covered=row['ci_low']<=exact<=row['ci_high'])
            rows.append(row)
grid=save_table(pd.DataFrame(rows),'european_parameter_grid')
display(grid[['kind','strike','maturity','estimate','benchmark','se','error','covered']])
fig,ax=plt.subplots()
for method,g in european[european.kind=='call'].groupby('method'):
    ax.errorbar(method,g.estimate.iloc[0],yerr=1.96*g.se.iloc[0],fmt='o',capsize=5)
ax.axhline(bsm_price(BASE),color='black',ls='--',label='BSM')
ax.set(ylabel='Call value',title='European call estimates: approximate 95% intervals')
ax.legend(); finish_figure(fig,'european_intervals')

## 6. Repeated-run convergence and efficiency

Each replication has its own deterministic random stream. Control coefficients are re-estimated for every replication. Runs at different budgets are independent; this is not a nested cumulative-path plot. RMSE is measured against BSM, and bias is the mean signed error.

A finite number of replications makes RMSE, coverage, and runtime estimates noisy. Coverage intervals are exact binomial intervals for the observed hit rate; the nominal 95% target need not be achieved in a small experiment. No p-value here is a model-approval rule.

In [ ]:
records=[]
for ni,n in enumerate(CFG['convergence_n']):
    for mi,method in enumerate(METHODS):
        for rep in range(CFG['repeats']):
            row=european_estimate(BASE,method,n,rng_for(20,ni,mi,rep),CFG['pilot'])
            row.update(budget=n,replication=rep,error=row['estimate']-bsm_price(BASE))
            row['covered']=row['ci_low']<=bsm_price(BASE)<=row['ci_high']
            records.append(row)
replications=save_table(pd.DataFrame(records),'european_replications')
summary=[]
for (n,method),g in replications.groupby(['budget','method']):
    coverage=binomtest(int(g.covered.sum()),len(g)).proportion_ci()
    variance=g.estimate.var(ddof=1)
    summary.append(dict(budget=n,method=method,bias=g.error.mean(),
       rmse=np.sqrt(np.mean(g.error**2)),empirical_sd=np.sqrt(variance),
       mean_reported_se=g.se.mean(),coverage=g.covered.mean(),
       coverage_low=coverage.low,coverage_high=coverage.high,
       mean_seconds=g.seconds.mean(),variance_time=variance*g.seconds.mean(),
       mean_total_evals=g.total_evals.mean()))
convergence=save_table(pd.DataFrame(summary),'convergence_summary')
display(convergence)
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for method,g in convergence.groupby('method'):
    axes[0].loglog(g.budget,g.rmse,'o-',label=method)
    axes[1].plot(g.budget,g.coverage,'o-',label=method)
ref=convergence[convergence.method=='plain'].sort_values('budget')
axes[0].loglog(ref.budget,ref.rmse.iloc[0]*np.sqrt(ref.budget.iloc[0]/ref.budget),'k--',label='N^-1/2 reference')
axes[0].set(xlabel='Production payoff evaluations',ylabel='RMSE versus BSM',title='Repeated-run convergence')
axes[1].axhline(.95,color='black',ls='--')
axes[1].set(xlabel='Production payoff evaluations',ylabel='Observed coverage',ylim=(0,1.03),title='Approximate 95% interval coverage')
for ax in axes: ax.legend()
finish_figure(fig,'convergence_and_coverage')

## 7. Sensitivity validation with common random numbers

Use central differences with the **same** normal draws in every perturbed valuation. Form a per-draw finite-difference contribution first; its sample mean and SE preserve the covariance across bumps. Rate bumps affect both dynamics and discounting. Theta negates the maturity derivative.

Common random numbers reduce noise but do not remove finite-difference bias. Gamma is particularly sensitive to payoff kinks and small bumps. Report sampling intervals and bump-size sensitivity separately; these intervals do not account for bump bias. A small SE is not proof of accuracy.

In [ ]:
def fd_samples(c,z,greek,h):
    if h<=0:
        raise ValueError('Positive bump required')
    field={'delta':'spot','gamma':'spot','vega':'vol','rho':'rate','theta':'maturity'}[greek]
    value=getattr(c,field)
    up=discounted_payoff(replace(c,**{field:value+h}),z)
    down=discounted_payoff(replace(c,**{field:value-h}),z)
    if greek=='gamma':
        return (up-2*discounted_payoff(c,z)+down)/h**2
    sign=-1 if greek=='theta' else 1
    return sign*(up-down)/(2*h)

rows=[]
for ki,kind in enumerate(('call','put')):
    c=replace(BASE,kind=kind)
    analytic=bsm_greeks(c)
    z=rng_for(30,ki).normal(size=CFG['greek_n'])
    bumps={'delta':[.1,.5,1.], 'gamma':[.1,.5,1.,2.],
           'vega':[.0005,.002,.01], 'rho':[.0001,.001,.01],
           'theta':[.0005,.002,.01]}
    for greek,hs in bumps.items():
        for h in hs:
            row=summarize(fd_samples(c,z,greek,h))
            row.update(kind=kind,greek=greek,bump=h,benchmark=analytic[greek],
                       error=row['estimate']-analytic[greek])
            rows.append(row)
greeks=save_table(pd.DataFrame(rows),'greek_bump_sensitivity')
display(greeks[['kind','greek','bump','estimate','benchmark','se','error']])
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax,greek in zip(axes,['delta','gamma']):
    g=greeks[(greeks.kind=='call') & (greeks.greek==greek)]
    ax.errorbar(g.bump,g.estimate,yerr=1.96*g.se,fmt='o-',capsize=4)
    ax.axhline(bsm_greeks(BASE)[greek],color='black',ls='--',label='Analytical')
    ax.set(xlabel='Absolute spot bump',ylabel=greek,title=f'Call {greek}: bump dependence')
    ax.legend()
finish_figure(fig,'greek_bumps')

## 8. Discrete arithmetic Asian options

For $A=m^{-1}\sum_{j=1}^m S_{jT/m}$ and $G=\exp[m^{-1}\sum_j\log S_{jT/m}]$, simulate exact transitions at the specified monitoring dates. A streaming implementation stores current log spot and sums, not the full path matrix.

The geometric average has log mean $\mu=\log S_0+(r-q-\sigma^2/2)\bar t$ and log variance
$$v=\frac{\sigma^2}{m^2}\sum_{i,j}\min(t_i,t_j).$$
Its discounted option value follows a lognormal closed form. This is an **exact benchmark for the geometric contract**, not the arithmetic contract. Use its discounted payoff as a control for the arithmetic payoff, with a separately estimated pilot coefficient.

For equal schedule and strike, $A\geq G$ pathwise; thus arithmetic call payoffs dominate geometric calls and arithmetic puts are no greater than geometric puts. These are useful checks, not a claim that Asian prices are always below European prices.

In [ ]:
def monitoring_times(c,m):
    if not isinstance(m,(int,np.integer)) or m<1:
        raise ValueError('m must be a positive integer')
    if c.maturity<=0:
        raise ValueError('Asian study requires positive maturity')
    return np.arange(1,m+1)*c.maturity/m

def geometric_price(c,m):
    ts=monitoring_times(c,m)
    mu=np.log(c.spot)+(c.rate-c.dividend-.5*c.vol**2)*ts.mean()
    variance=c.vol**2*np.minimum.outer(ts,ts).sum()/m**2
    discount=np.exp(-c.rate*c.maturity)
    if variance==0:
        return float(discount*payoff(np.exp(mu),c))
    sd=np.sqrt(variance)
    d2=(mu-np.log(c.strike))/sd
    d1=d2+sd
    eg=np.exp(mu+.5*variance)
    return float(discount*(eg*norm.cdf(d1)-c.strike*norm.cdf(d2)) if c.kind=='call'
                 else discount*(c.strike*norm.cdf(-d2)-eg*norm.cdf(-d1)))

def asian_draws(c,n,m,rng,antithetic=False):
    monitoring_times(c,m)
    dt=c.maturity/m
    log_s=np.full(n,np.log(c.spot))
    arithmetic_sum=np.zeros(n)
    log_sum=np.zeros(n)
    if antithetic:
        log_a=log_s.copy(); sum_a=np.zeros(n); logs_a=np.zeros(n)
    for _ in range(m):
        z=rng.standard_normal(n)
        drift=(c.rate-c.dividend-.5*c.vol**2)*dt
        shock=c.vol*np.sqrt(dt)*z
        log_s+=drift+shock
        arithmetic_sum+=np.exp(log_s); log_sum+=log_s
        if antithetic:
            log_a+=drift-shock
            sum_a+=np.exp(log_a); logs_a+=log_a
    d=np.exp(-c.rate*c.maturity)
    y=d*payoff(arithmetic_sum/m,c)
    x=d*payoff(np.exp(log_sum/m),c)
    if antithetic:
        y=.5*(y+d*payoff(sum_a/m,c))
        x=.5*(x+d*payoff(np.exp(logs_a/m),c))
    return y,x

def asian_estimate(c,method,budget,m,rng,pilot):
    if method not in METHODS or budget<4 or pilot<2:
        raise ValueError('Invalid method or sample budget')
    start=perf_counter(); beta=np.nan; extra=0
    if method=='control':
        yp,xp=asian_draws(c,pilot,m,rng)
        beta=fit_beta(yp,xp); extra=pilot
    y,x=asian_draws(c,budget//2 if method=='antithetic' else budget,m,rng,
                    antithetic=(method=='antithetic'))
    geo=summarize(x)
    if method=='control':
        y=y-beta*(x-geometric_price(c,m))
    result=summarize(y)
    result.update(method=method,beta=beta,monitoring=m,production_evals=budget if method!='antithetic' else 2*(budget//2),
                  pilot_evals=extra,geometric_estimate=geo['estimate'],geometric_se=geo['se'],
                  geometric_benchmark=geometric_price(c,m),seconds=perf_counter()-start)
    return result

rows=[]
for ki,kind in enumerate(('call','put')):
    c=replace(BASE,kind=kind)
    y,x=asian_draws(c,2000,CFG['monitoring'],rng_for(40,ki))
    check(f'asian_pathwise_order_{kind}',np.all(y>=x-1e-10) if kind=='call' else np.all(y<=x+1e-10))
    check(f'one_monitor_equals_european_{kind}',np.isclose(geometric_price(c,1),bsm_price(c),atol=1e-10))
    for mi,method in enumerate(METHODS):
        row=asian_estimate(c,method,CFG['asian_n'],CFG['monitoring'],rng_for(41,ki,mi),CFG['asian_pilot'])
        row['kind']=kind
        rows.append(row)
asian=save_table(pd.DataFrame(rows),'asian_comparison')
display(asian)
save_table(pd.DataFrame(checks),'validation_checks')

### Monitoring schedule and repeated-seed stability

The following table compares separately defined monitoring contracts. Do not call differences between 12, 52 and 252 dates discretisation bias against a single discrete contract. No continuous-monitoring extrapolation is claimed.

The stability experiment below reports variation across independent runs. There is no exact arithmetic-Asian benchmark here; dispersion is not a proof of absence of model or implementation error.

In [ ]:
rows=[]
for m in (12,52,252):
    row=asian_estimate(BASE,'control',CFG['asian_n'],m,rng_for(42,m),CFG['asian_pilot'])
    rows.append(row)
display(save_table(pd.DataFrame(rows),'asian_monitoring_contracts'))
rows=[]
for mi,method in enumerate(METHODS):
    for rep in range(8 if MODE=='quick' else 40):
        row=asian_estimate(BASE,method,CFG['asian_n'],CFG['monitoring'],rng_for(43,mi,rep),CFG['asian_pilot'])
        row['replication']=rep
        rows.append(row)
asian_runs=save_table(pd.DataFrame(rows),'asian_replications')
asian_summary=asian_runs.groupby('method').agg(mean_estimate=('estimate','mean'),empirical_sd=('estimate','std'),mean_se=('se','mean'),mean_seconds=('seconds','mean')).reset_index()
display(save_table(asian_summary,'asian_stability_summary'))
fig,ax=plt.subplots()
for ki,kind in enumerate(('call','put')):
    g=asian[asian.kind==kind]
    x=np.arange(len(g))+(.08 if ki else -.08)
    ax.errorbar(x,g.estimate,yerr=1.96*g.se,fmt='o',capsize=4,label=kind)
ax.set_xticks(np.arange(3),METHODS)
ax.set(ylabel='Arithmetic Asian value',title='Asian estimators: approximate 95% intervals')
ax.legend(); finish_figure(fig,'asian_intervals')

## 9. Interpretation checklist

Review generated tables before writing a publication abstract:

1. Do all deterministic gates pass? Investigate failures before increasing simulation counts.
2. How do benchmark errors compare with reported SEs across contracts, not just the at-the-money example?
3. Does repeated-run RMSE decline with budget? Does empirical dispersion resemble reported SE?
4. Does variance reduction remain beneficial after pilot overhead and noisy runtime measurements?
5. Which Greeks are stable across bumps? Separate finite-difference bias from sampling uncertainty.
6. Do geometric Asian checks and arithmetic/geometric payoff ordering hold?
7. Are sparse-payoff cases producing a suspicious zero SE? Expand the design rather than reporting certainty.

### Limitations

- GBM/BSM is a controlled model, not an empirical fit to market smiles, jumps, liquidity or volatility dynamics.
- Dividend yield is continuous and deterministic; actual discrete dividends and corporate actions are outside scope.
- European sensitivities are local model derivatives, not hedge-performance results.
- Approximate intervals can under-cover with skewness or rare payoffs. Individual misses are expected; zero variance in a finite sample is not evidence of zero risk.
- Pilot coefficient uncertainty is represented across replications, while each reported SE is conditional on that pilot.
- Asian standard errors cover sampling uncertainty for a fixed monitoring contract, not calibration uncertainty or model risk.
- Runtime is hardware-dependent; this study is not a stable latency benchmark.
- No American exercise, Heston, jump calibration, VaR/ES framework, live trading, or production deployment is claimed.
- This notebook is research code. Production use would require independent review, broader testing, data/model governance, operational controls and monitoring.

### References for method verification

- Black and Scholes (1973), The Pricing of Options and Corporate Liabilities.
- Merton (1973), Theory of Rational Option Pricing.
- Glasserman (2004), Monte Carlo Methods in Financial Engineering: simulation, variance reduction and sensitivities.
- The discrete-geometric benchmark is derived from the joint normality of monitored log prices as shown above; it must use exactly the stated schedule.

These are references, not additions to the user's active reading list.

In [ ]:
# Export configuration, environment and generated numerical observations.
versions={}
for package in ('numpy','pandas','scipy','matplotlib','nbformat'):
    try:
        versions[package]=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package]='not installed'
manifest=dict(utc_timestamp=datetime.now(timezone.utc).isoformat(),mode=MODE,
              config=CFG,base_contract=asdict(BASE),python=sys.version,
              platform=platform.platform(),versions=versions,
              notes='Synthetic GBM study; results are not market calibration or independent certification.')
(RESULTS/'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
observations={'deterministic_checks_passed':all(c['passed'] for c in checks),
              'deterministic_check_count':len(checks),
              'european_single_run_interval_hits':int(european.covered.sum()),
              'european_single_run_cases':len(european),
              'grid_interval_hits':int(grid.covered.sum()),'grid_cases':len(grid),
              'interpretation':'Interval misses alone are not deterministic check failures. Review replication and bump tables.'}
(RESULTS/'run_observations.json').write_text(json.dumps(observations,indent=2),encoding='utf-8')
artifacts=[]
for path in sorted(RESULTS.rglob('*')):
    if path.is_file() and path.name!='artifact_checksums.csv':
        artifacts.append({'path':str(path.relative_to(PROJECT_ROOT)),
                          'bytes':path.stat().st_size,
                          'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
pd.DataFrame(artifacts).to_csv(RESULTS/'artifact_checksums.csv',index=False)
display(pd.DataFrame([observations]))
display(pd.DataFrame(artifacts))


### **➤ SENSITIVTY DIVERGENCE**

### European sensitivities: analytical benchmarks and Monte Carlo uncertainty

This section compares all five European Greeks—delta, gamma, vega, theta,
and rho—for calls and puts across a spot-price grid.

The analytical BSM sensitivities are the reference values. Monte Carlo
sensitivities use central finite differences with common random numbers:
the same normal draws are used for the upward, downward, and base
valuations where required.

For each Greek, the finite-difference calculation produces one contribution
per independent normal draw. The estimate is the sample mean of those
contributions; its standard error therefore retains the dependence between
the bumped valuations.

#### Reporting conventions

| Greek | Interpretation | Units used here |
|---|---|---|
| Delta | Sensitivity to spot | Price change per unit change in spot |
| Gamma | Sensitivity of delta to spot | Price change per squared spot unit |
| Vega | Sensitivity to volatility | Price change per 1.00 change in annualised volatility |
| Theta | Sensitivity to calendar time elapsed | Price change per year elapsed |
| Rho | Sensitivity to the risk-free rate | Price change per 1.00 change in the continuously compounded annual rate |

Vega and rho are not divided by 100 in these plots. Theta is not divided
by 365. The conventions match the analytical functions and exported tables.

#### Two complementary diagnostics

1. **Spot-grid comparison:** hold strike, maturity, volatility, dividend
   yield, and rate fixed while changing spot. Compare the analytical
   sensitivity curve with Monte Carlo estimates and approximate 95%
   pointwise confidence intervals.

2. **Bump-size comparison:** at the base contract, vary the finite-difference
   bump while holding the random draws fixed. Compare the resulting estimates
   against the analytical Greek.

The spot-grid experiment uses the same random draws across spot levels
within each option type. This reduces visual simulation noise when comparing
neighbouring contracts, but makes their estimation errors correlated.
Intervals are pointwise, not simultaneous confidence bands.

#### Interpretation and limitations

Sampling uncertainty and finite-difference bias are different:

- The confidence interval describes Monte Carlo sampling uncertainty for
  the selected finite-difference estimator.
- It does not include finite-difference truncation bias.
- Smaller bumps do not necessarily produce better estimates. Payoff kinks,
  rare strike crossings, and floating-point cancellation can make estimates
  unstable, particularly for gamma.
- An analytical value outside one interval is not, by itself, proof of a
  coding error. Consider repeated runs, bump dependence, and the size of
  the experiment.
- A zero estimated standard error can occur when the simulated sample
  contains no relevant payoff changes; it must not be interpreted as
  certainty.

These are local sensitivities under BSM assumptions, not evidence of
realised hedge performance.

In [ ]:
# European Greek visualisation extension.
# Requires the notebook's setup, pricing functions, and Greek diagnostic cells.

required_objects = (
    "BASE",
    "CFG",
    "bsm_price",
    "bsm_greeks",
    "fd_samples",
    "summarize",
    "rng_for",
    "save_table",
    "finish_figure",
    "greeks",
)
missing = [name for name in required_objects if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the earlier notebook cells first. Missing: " + ", ".join(missing)
    )

if BASE.vol <= 0 or BASE.maturity <= 0:
    raise ValueError(
        "This sensitivity experiment requires positive volatility and maturity."
    )

GREEK_ORDER = ("delta", "gamma", "vega", "theta", "rho")

GREEK_LABELS = {
    "delta": "Delta — per spot unit",
    "gamma": "Gamma — per squared spot unit",
    "vega": "Vega — per unit volatility",
    "theta": "Theta — per year elapsed",
    "rho": "Rho — per unit annual rate",
}

# Declared before observing results; not selected to improve apparent fit.
SPOT_BUMP_FRACTION = {
    "delta": 0.005,
    "gamma": 0.01,
}
PARAMETER_BUMPS = {
    "vega": min(0.002, BASE.vol * 0.05),
    "theta": min(0.002, BASE.maturity * 0.05),
    "rho": 0.001,
}

spot_grid = BASE.strike * np.linspace(0.70, 1.30, 13)
dense_spot_grid = BASE.strike * np.linspace(0.70, 1.30, 201)

spot_rows = []
analytical_curve_rows = []

for kind_index, kind in enumerate(("call", "put")):
    # Shared across spot levels and Greeks within this option type.
    # Independent draws, without antithetic pairing or moment matching.
    z = rng_for(31, kind_index).standard_normal(CFG["greek_n"])

    for spot in spot_grid:
        contract = replace(BASE, kind=kind, spot=float(spot))
        analytical = bsm_greeks(contract)

        for greek in GREEK_ORDER:
            if greek in SPOT_BUMP_FRACTION:
                bump = contract.spot * SPOT_BUMP_FRACTION[greek]
            else:
                bump = PARAMETER_BUMPS[greek]

            samples = fd_samples(contract, z, greek, bump)
            result = summarize(samples)

            benchmark = float(analytical[greek])
            error = result["estimate"] - benchmark

            spot_rows.append(
                {
                    "kind": kind,
                    "greek": greek,
                    "spot": contract.spot,
                    "strike": contract.strike,
                    "moneyness": contract.spot / contract.strike,
                    "maturity": contract.maturity,
                    "volatility": contract.vol,
                    "rate": contract.rate,
                    "dividend": contract.dividend,
                    "bump": bump,
                    "benchmark": benchmark,
                    **result,
                    "error": error,
                    "absolute_error": abs(error),
                    "benchmark_in_interval": (
                        result["ci_low"] <= benchmark <= result["ci_high"]
                    ),
                }
            )

    for spot in dense_spot_grid:
        contract = replace(BASE, kind=kind, spot=float(spot))
        analytical_curve_rows.append(
            {
                "kind": kind,
                "spot": float(spot),
                **bsm_greeks(contract),
            }
        )

greek_spot_grid = pd.DataFrame(spot_rows)
greek_analytical_curves = pd.DataFrame(analytical_curve_rows)

numeric_columns = [
    "spot",
    "bump",
    "benchmark",
    "estimate",
    "se",
    "ci_low",
    "ci_high",
    "error",
]
if not np.isfinite(
    greek_spot_grid[numeric_columns].to_numpy(dtype=float)
).all():
    raise FloatingPointError("Non-finite values in the Greek spot-grid experiment.")

save_table(greek_spot_grid, "greeks_spot_grid")
save_table(greek_analytical_curves, "greeks_analytical_spot_curves")

display(
    greek_spot_grid[
        [
            "kind",
            "greek",
            "spot",
            "bump",
            "estimate",
            "benchmark",
            "se",
            "error",
            "benchmark_in_interval",
        ]
    ].round(6)
)

# Analytical curves with Monte Carlo point estimates and pointwise intervals.
for kind in ("call", "put"):
    fig, axes = plt.subplots(3, 2, figsize=(13, 12))
    flat_axes = axes.ravel()

    for ax, greek in zip(flat_axes, GREEK_ORDER):
        observed = greek_spot_grid[
            (greek_spot_grid["kind"] == kind)
            & (greek_spot_grid["greek"] == greek)
        ].sort_values("spot")

        reference = greek_analytical_curves[
            greek_analytical_curves["kind"] == kind
        ].sort_values("spot")

        interval_widths = np.vstack(
            [
                observed["estimate"] - observed["ci_low"],
                observed["ci_high"] - observed["estimate"],
            ]
        )

        ax.plot(
            reference["spot"],
            reference[greek],
            color="black",
            linewidth=1.8,
            label="Analytical BSM",
        )
        ax.errorbar(
            observed["spot"],
            observed["estimate"],
            yerr=interval_widths,
            fmt="o",
            markersize=4,
            capsize=3,
            color="tab:blue",
            label="MC finite difference: 95% interval",
        )
        ax.axvline(
            BASE.strike,
            color="grey",
            linestyle=":",
            linewidth=1,
            label="Strike",
        )
        ax.set(
            xlabel="Spot price",
            ylabel=GREEK_LABELS[greek],
            title=f"{kind.capitalize()} {greek}",
        )
        ax.legend(fontsize=8)

    flat_axes[-1].axis("off")
    flat_axes[-1].text(
        0.02,
        0.95,
        (
            f"Independent draws per estimate: {CFG['greek_n']:,}\n"
            f"Strike: {BASE.strike:g}\n"
            f"Maturity: {BASE.maturity:g} years\n"
            f"Volatility: {BASE.vol:.2%}\n"
            f"Rate: {BASE.rate:.2%}\n"
            f"Dividend yield: {BASE.dividend:.2%}\n\n"
            "Intervals are pointwise and exclude bump bias.\n"
            "Random draws are shared across spot levels."
        ),
        transform=flat_axes[-1].transAxes,
        va="top",
        fontsize=10,
        linespacing=1.6,
    )

    fig.suptitle(
        f"European {kind}: analytical and Monte Carlo sensitivities",
        fontsize=14,
        y=1.01,
    )
    finish_figure(fig, f"european_{kind}_all_greeks_vs_spot")


# Signed errors make deviations easier to see than overlapping price-scale curves.
fig, axes = plt.subplots(3, 2, figsize=(13, 12))
flat_axes = axes.ravel()

for ax, greek in zip(flat_axes, GREEK_ORDER):
    for kind, colour, marker in (
        ("call", "tab:blue", "o"),
        ("put", "tab:orange", "s"),
    ):
        observed = greek_spot_grid[
            (greek_spot_grid["kind"] == kind)
            & (greek_spot_grid["greek"] == greek)
        ].sort_values("spot")

        error_lower = observed["ci_low"] - observed["benchmark"]
        error_upper = observed["ci_high"] - observed["benchmark"]

        ax.errorbar(
            observed["spot"],
            observed["error"],
            yerr=np.vstack(
                [
                    observed["error"] - error_lower,
                    error_upper - observed["error"],
                ]
            ),
            fmt=marker + "-",
            markersize=3,
            capsize=2,
            alpha=0.8,
            color=colour,
            label=kind.capitalize(),
        )

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set(
        xlabel="Spot price",
        ylabel=f"MC minus analytical {greek}",
        title=f"{greek.capitalize()}: signed estimation error",
    )
    ax.legend()

flat_axes[-1].axis("off")
flat_axes[-1].text(
    0.02,
    0.95,
    (
        "Error bars are sampling intervals shifted\n"
        "by the analytical benchmark.\n\n"
        "Errors across spot levels are correlated.\n"
        "Do not interpret this as independent\n"
        "hypothesis testing across contracts."
    ),
    transform=flat_axes[-1].transAxes,
    va="top",
    linespacing=1.6,
)
finish_figure(fig, "european_all_greeks_signed_errors")


# Extend the existing base-contract bump diagnostics to all five Greeks.
required_bump_columns = {
    "kind",
    "greek",
    "bump",
    "estimate",
    "benchmark",
    "ci_low",
    "ci_high",
}
if not required_bump_columns.issubset(greeks.columns):
    raise ValueError("The earlier 'greeks' table has an unexpected schema.")

for kind in ("call", "put"):
    fig, axes = plt.subplots(3, 2, figsize=(13, 12))
    flat_axes = axes.ravel()

    for ax, greek in zip(flat_axes, GREEK_ORDER):
        observed = greeks[
            (greeks["kind"] == kind)
            & (greeks["greek"] == greek)
        ].sort_values("bump")

        if observed.empty:
            raise ValueError(f"No bump diagnostics found for {kind} {greek}.")

        ax.errorbar(
            observed["bump"],
            observed["estimate"],
            yerr=np.vstack(
                [
                    observed["estimate"] - observed["ci_low"],
                    observed["ci_high"] - observed["estimate"],
                ]
            ),
            fmt="o-",
            markersize=4,
            capsize=4,
            color="tab:blue",
            label="MC finite difference: 95% interval",
        )
        ax.axhline(
            observed["benchmark"].iloc[0],
            color="black",
            linestyle="--",
            label="Analytical BSM",
        )
        ax.set_xscale("log")
        ax.set(
            xlabel="Absolute finite-difference bump (log scale)",
            ylabel=GREEK_LABELS[greek],
            title=f"{kind.capitalize()} {greek}: bump diagnostic",
        )
        ax.legend(fontsize=8)

    flat_axes[-1].axis("off")
    flat_axes[-1].text(
        0.02,
        0.95,
        (
            "Base contract only.\n\n"
            "Spot bumps: delta and gamma.\n"
            "Volatility bumps: vega.\n"
            "Maturity bumps in years: theta.\n"
            "Annual-rate bumps: rho.\n\n"
            "A smaller bump is not automatically better."
        ),
        transform=flat_axes[-1].transAxes,
        va="top",
        linespacing=1.6,
    )
    finish_figure(fig, f"european_{kind}_all_greeks_bump_diagnostics")


# Descriptive summaries, not pass/fail model-approval thresholds.
greek_grid_summary = (
    greek_spot_grid.groupby(["kind", "greek"], sort=False)
    .agg(
        contracts=("spot", "size"),
        mean_absolute_error=("absolute_error", "mean"),
        maximum_absolute_error=("absolute_error", "max"),
        mean_standard_error=("se", "mean"),
        benchmark_interval_hits=("benchmark_in_interval", "sum"),
        zero_standard_error_cases=("zero_se", "sum"),
    )
    .reset_index()
)

save_table(greek_grid_summary, "greeks_spot_grid_summary")
display(greek_grid_summary)


### **➤ AMERICAN OPTIONS**

## American calls and puts: exercise policies and independent validation

### Objective

Estimate early-exercise option values under the same risk-neutral GBM model
used elsewhere in this notebook, and examine where the fitted policy chooses
to exercise.

The simulation permits exercise on a finite, equally spaced grid, including
time zero and maturity. It therefore implements a Bermudan approximation to
American exercise.

### Longstaff–Schwartz policy

At each intermediate exercise date:

1. Select training paths that are in the money.
2. Regress their discounted future realised cash flows on polynomial functions
   of the current stock price.
3. Compare immediate exercise value with fitted continuation value.
4. Replace the future cash flow when immediate exercise is preferred.

The polynomial state is scaled before regression. Regressions with too few
observations, deficient rank, or extreme conditioning are skipped; the policy
continues at those dates. Such skips are recorded.

The fitted policy is evaluated forward on an independent set of paths.
Each path produces one stopping time and one discounted cash flow.

### Time-zero decision and fallback

The choice between immediate exercise, the fitted LSMC policy, and holding
to maturity is made using training information only. The evaluation sample
does not select the winning policy.

The European fallback helps detect an unreliable exercise policy, but does
not guarantee the selected policy is optimal. Selecting on training data can
still overfit; independent valuation exposes rather than removes that problem.

### Benchmarks

A separately implemented Cox–Ross–Rubinstein tree provides:

- An American tree value: exercise at every tree date.
- A matched-schedule Bermudan value: exercise only on the LSMC exercise dates.
- A European tree value: exercise only at maturity.

The tree uses four substeps per LSMC interval by default. It is a numerical
benchmark, not an exact oracle. A separate refinement table examines its
stability.

The BSM European price and immediate intrinsic value provide additional
reference values.

### Uncertainty and interpretation

- Evaluation paths are independent conditional on the fitted policy.
- The reported interval covers Monte Carlo sampling uncertainty for that policy.
- It does not cover exercise-grid error, regression approximation,
  training-policy uncertainty, tree error, or model risk.
- The expected payoff of an admissible fixed policy cannot exceed the optimal
  value. A finite Monte Carlo estimate may exceed a numerical benchmark by chance.
- A fitted policy can underperform holding to maturity out of sample.
  Do not clip or overwrite that result.
- A non-dividend-paying call with a nonnegative interest rate supplies a useful
  no-early-exercise diagnostic. Dividend-paying calls are examined separately.

### Visualisations

1. Evaluation paths with their actual exercise decisions marked.
2. Exercise time versus stock price at exercise.
3. Early-exercise frequency conditional on being alive within a time/spot bin.
4. Policy-value comparisons against analytical and tree benchmarks.
5. Exercise-grid and regression-degree sensitivity.

The scatter plot and heatmap describe the learned policy on sampled states.
They are not exact optimal exercise boundaries.

In [ ]:
# Requires the existing Contract, payoff, summarize, rng_for, and export helpers.

AMERICAN_CFG = {
    "train_paths": 15000 if MODE == "quick" else 80000,
    "evaluation_paths": 20000 if MODE == "quick" else 100000,
    "exercise_steps": 50,
    "degree": 2,
    "tree_substeps": 4,
    "min_regression": 30,
}

def exercise_paths(contract, n_paths, n_steps, rng):
    if contract.maturity <= 0 or contract.vol <= 0:
        raise ValueError("This module requires positive maturity and volatility.")
    if n_paths < 2 or n_steps < 2:
        raise ValueError("Need at least two paths and two exercise intervals.")

    dt = contract.maturity / n_steps
    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = contract.spot

    drift = (
        contract.rate - contract.dividend - 0.5 * contract.vol**2
    ) * dt
    diffusion = contract.vol * np.sqrt(dt)

    for step in range(1, n_steps + 1):
        z = rng.standard_normal(n_paths)
        paths[:, step] = paths[:, step - 1] * np.exp(
            drift + diffusion * z
        )

    return paths


def polynomial_features(state, degree):
    return np.polynomial.polynomial.polyvander(state, degree)


def fit_exercise_policy(contract, paths, degree=2, minimum=30):
    if degree < 1:
        raise ValueError("Polynomial degree must be positive.")

    n_paths, columns = paths.shape
    steps = columns - 1
    dt = contract.maturity / steps

    cashflow = payoff(paths[:, -1], contract)
    stopping_step = np.full(n_paths, steps, dtype=int)

    models = {}
    diagnostics = []

    for step in range(steps - 1, 0, -1):
        intrinsic = payoff(paths[:, step], contract)
        itm = intrinsic > 0
        count = int(itm.sum())

        record = {
            "step": step,
            "time": step * dt,
            "itm_training_paths": count,
            "status": "insufficient_observations",
            "condition_number": np.nan,
        }

        if count < max(minimum, 5 * (degree + 1)):
            diagnostics.append(record)
            continue

        x = paths[itm, step] / contract.strike
        centre = float(x.mean())
        scale = float(x.std(ddof=1))

        if scale < 1e-10:
            record["status"] = "degenerate_state"
            diagnostics.append(record)
            continue

        design = polynomial_features((x - centre) / scale, degree)
        target = cashflow[itm] * np.exp(
            -contract.rate * dt * (stopping_step[itm] - step)
        )

        condition = float(np.linalg.cond(design))
        coefficients, _, rank, _ = np.linalg.lstsq(
            design, target, rcond=None
        )

        record["condition_number"] = condition

        if rank < degree + 1 or not np.isfinite(condition) or condition > 1e10:
            record["status"] = "unstable_regression"
            diagnostics.append(record)
            continue

        models[step] = {
            "centre": centre,
            "scale": scale,
            "coefficients": coefficients,
        }

        continuation = np.maximum(design @ coefficients, 0.0)
        exercise = intrinsic[itm] > continuation

        selected = np.flatnonzero(itm)[exercise]
        cashflow[selected] = intrinsic[selected]
        stopping_step[selected] = step

        record["status"] = "fitted"
        diagnostics.append(record)

    fitted_training_value = float(
        np.mean(
            cashflow
            * np.exp(-contract.rate * stopping_step * dt)
        )
    )
    european_training_value = float(
        np.mean(
            np.exp(-contract.rate * contract.maturity)
            * payoff(paths[:, -1], contract)
        )
    )
    immediate = float(payoff(contract.spot, contract))

    # Select on training information only; never maximise evaluation estimates.
    training_candidates = {
        "lsmc": fitted_training_value,
        "european": european_training_value,
        "immediate": immediate,
    }
    selected_policy = max(
        training_candidates, key=training_candidates.get
    )

    return {
        "steps": steps,
        "degree": degree,
        "models": models,
        "selected_policy": selected_policy,
        "training_candidates": training_candidates,
        "diagnostics": pd.DataFrame(diagnostics),
    }


def evaluate_exercise_policy(contract, paths, policy):
    n_paths, columns = paths.shape
    steps = columns - 1

    if steps != policy["steps"]:
        raise ValueError("Training and evaluation exercise schedules differ.")

    dt = contract.maturity / steps
    selected_policy = policy["selected_policy"]

    stopping_step = np.full(n_paths, steps, dtype=int)
    cashflow = payoff(paths[:, -1], contract).copy()

    if selected_policy == "immediate":
        stopping_step[:] = 0
        cashflow[:] = payoff(contract.spot, contract)

    elif selected_policy == "lsmc":
        alive = np.ones(n_paths, dtype=bool)

        for step in range(1, steps):
            model = policy["models"].get(step)
            if model is None:
                continue

            intrinsic = payoff(paths[:, step], contract)
            eligible = alive & (intrinsic > 0)
            indices = np.flatnonzero(eligible)

            if len(indices) == 0:
                continue

            state = (
                paths[indices, step] / contract.strike - model["centre"]
            ) / model["scale"]

            design = polynomial_features(state, policy["degree"])
            continuation = np.maximum(
                design @ model["coefficients"], 0.0
            )
            exercised = indices[
                intrinsic[indices] > continuation
            ]

            stopping_step[exercised] = step
            cashflow[exercised] = intrinsic[exercised]
            alive[exercised] = False

    elif selected_policy != "european":
        raise ValueError("Unknown exercise policy.")

    discounted = cashflow * np.exp(
        -contract.rate * stopping_step * dt
    )
    european_discounted = (
        np.exp(-contract.rate * contract.maturity)
        * payoff(paths[:, -1], contract)
    )

    # Shared-path difference: do not add separate standard errors.
    premium = summarize(discounted - european_discounted)

    return {
        "summary": summarize(discounted),
        "premium_summary": premium,
        "stopping_step": stopping_step,
        "cashflow": cashflow,
        "discounted_cashflow": discounted,
        "european_discounted_cashflow": european_discounted,
    }

In [ ]:
def crr_value(contract, tree_steps, exercise_every=None):
    """
    exercise_every=None: European exercise only.
    exercise_every=1: American exercise on every tree date.
    Other positive integers: restricted exercise schedule.
    """
    if not isinstance(tree_steps, (int, np.integer)) or tree_steps < 2:
        raise ValueError("tree_steps must be an integer >= 2.")

    if exercise_every is not None:
        if exercise_every < 1 or tree_steps % exercise_every != 0:
            raise ValueError("Exercise interval must divide tree_steps.")

    dt = contract.maturity / tree_steps
    up = np.exp(contract.vol * np.sqrt(dt))
    down = 1.0 / up

    probability = (
        np.exp((contract.rate - contract.dividend) * dt) - down
    ) / (up - down)

    if not 0.0 <= probability <= 1.0:
        raise ValueError("Invalid CRR probability; refine the tree.")

    indices = np.arange(tree_steps + 1)
    stock = contract.spot * np.exp(
        (2 * indices - tree_steps) * np.log(up)
    )
    values = payoff(stock, contract)

    for step in range(tree_steps - 1, -1, -1):
        values = np.exp(-contract.rate * dt) * (
            probability * values[1:]
            + (1.0 - probability) * values[:-1]
        )

        if exercise_every is not None and step % exercise_every == 0:
            indices = np.arange(step + 1)
            stock = contract.spot * np.exp(
                (2 * indices - step) * np.log(up)
            )
            values = np.maximum(values, payoff(stock, contract))

    return float(values[0])


# Deterministic structural checks for the independently implemented tree.
for kind in ("call", "put"):
    contract = replace(BASE, kind=kind)
    european_tree = crr_value(contract, 400)
    american_tree = crr_value(contract, 400, exercise_every=1)
    bermudan_tree = crr_value(contract, 400, exercise_every=8)

    assert american_tree + 1e-10 >= bermudan_tree
    assert bermudan_tree + 1e-10 >= european_tree
    assert american_tree + 1e-10 >= float(payoff(contract.spot, contract))

# No-dividend, nonnegative-rate diagnostic.
no_dividend_call = replace(
    BASE, kind="call", dividend=0.0, rate=max(BASE.rate, 0.0)
)
assert np.isclose(
    crr_value(no_dividend_call, 400, exercise_every=1),
    crr_value(no_dividend_call, 400),
    atol=1e-8,
)

AMERICAN_CASES = {
    "put_no_dividend": replace(BASE, kind="put", dividend=0.0),
    "call_no_dividend": replace(
        BASE, kind="call", dividend=0.0, rate=max(BASE.rate, 0.0)
    ),
    "call_with_dividend": replace(
        BASE, kind="call", dividend=0.08
    ),
}

american_runs = {}
pricing_rows = []
policy_diagnostic_frames = []

for case_index, (case_name, contract) in enumerate(AMERICAN_CASES.items()):
    steps = AMERICAN_CFG["exercise_steps"]

    training_paths = exercise_paths(
        contract,
        AMERICAN_CFG["train_paths"],
        steps,
        rng_for(60, case_index, 0),
    )
    policy = fit_exercise_policy(
        contract,
        training_paths,
        degree=AMERICAN_CFG["degree"],
        minimum=AMERICAN_CFG["min_regression"],
    )
    del training_paths

    evaluation_paths = exercise_paths(
        contract,
        AMERICAN_CFG["evaluation_paths"],
        steps,
        rng_for(60, case_index, 1),
    )
    evaluation = evaluate_exercise_policy(
        contract, evaluation_paths, policy
    )

    stopping = evaluation["stopping_step"]
    paid_early = (stopping < steps) & (evaluation["cashflow"] > 0)

    tree_steps = steps * AMERICAN_CFG["tree_substeps"]
    row = {
        "case": case_name,
        "selected_policy": policy["selected_policy"],
        **evaluation["summary"],
        "european_bsm": bsm_price(contract),
        "european_tree": crr_value(contract, tree_steps),
        "bermudan_tree": crr_value(
            contract,
            tree_steps,
            exercise_every=AMERICAN_CFG["tree_substeps"],
        ),
        "american_tree": crr_value(
            contract, tree_steps, exercise_every=1
        ),
        "intrinsic": float(payoff(contract.spot, contract)),
        "early_paid_fraction": float(paid_early.mean()),
        "premium_vs_hold_estimate": evaluation["premium_summary"]["estimate"],
        "premium_vs_hold_se": evaluation["premium_summary"]["se"],
        "exercise_steps": steps,
        "regression_degree": policy["degree"],
        "fitted_dates": len(policy["models"]),
    }
    pricing_rows.append(row)

    diagnostics = policy["diagnostics"].copy()
    diagnostics.insert(0, "case", case_name)
    policy_diagnostic_frames.append(diagnostics)

    events = pd.DataFrame({
        "path_id": np.arange(len(stopping)),
        "stopping_step": stopping,
        "stopping_time": stopping * contract.maturity / steps,
        "stock_at_stopping": evaluation_paths[
            np.arange(len(stopping)), stopping
        ],
        "cashflow": evaluation["cashflow"],
        "discounted_cashflow": evaluation["discounted_cashflow"],
        "paid_early": paid_early,
    })
    save_table(events, f"{case_name}_exercise_events")

    american_runs[case_name] = {
        "contract": contract,
        "paths": evaluation_paths,
        "policy": policy,
        "evaluation": evaluation,
        "events": events,
    }

american_prices = save_table(
    pd.DataFrame(pricing_rows), "american_policy_comparison"
)
save_table(
    pd.concat(policy_diagnostic_frames, ignore_index=True),
    "american_regression_diagnostics",
)
display(american_prices)

fig, axes = plt.subplots(
    1, len(AMERICAN_CASES), figsize=(15, 4.5)
)
for ax, row in zip(axes, pricing_rows):
    ax.errorbar(
        0,
        row["estimate"],
        yerr=np.array([
            [row["estimate"] - row["ci_low"]],
            [row["ci_high"] - row["estimate"]],
        ]),
        fmt="o",
        capsize=5,
        color="tab:blue",
        label="Selected policy: 95% interval",
    )
    for value, colour, style, label in (
        (row["european_bsm"], "grey", ":", "European BSM"),
        (row["bermudan_tree"], "tab:orange", "--", "Matched-grid tree"),
        (row["american_tree"], "black", "-", "American tree"),
    ):
        ax.axhline(value, color=colour, ls=style, label=label)

    ax.set(
        title=row["case"].replace("_", " "),
        ylabel="Option value",
        xticks=[],
    )
    ax.legend(fontsize=8)

finish_figure(fig, "american_policy_benchmarks")

### Reading the exercise visualisations

The paths and exercise markers below come from the same independent
evaluation simulation. Paths stop visually when the policy stops, rather
than showing later stock movements as though the option were still alive.

Only strictly positive early-exercise cash flows appear in the event cloud.
Maturity payoffs are not labelled early exercise.

The heatmap reports:

    positive-payoff early exercises in a time/spot bin
    ------------------------------------------------
    paths alive before the decision in that time/spot bin

The denominator includes both in-the-money and out-of-the-money surviving
paths. A minimum occupancy requirement suppresses poorly supported bins.
The upper tail is outside the displayed range; it is not removed from pricing.

A high-frequency region indicates what this fitted policy did in sufficiently
populated sampled states. It is not a proven optimal exercise region.

In [ ]:
from matplotlib.colors import Normalize

exercise_heatmap_tables = []

for case_index, (case_name, run) in enumerate(american_runs.items()):
    contract = run["contract"]
    paths = run["paths"]
    events = run["events"]
    steps = paths.shape[1] - 1
    times = np.linspace(0, contract.maturity, steps + 1)

    early_ids = events.loc[events["paid_early"], "path_id"].to_numpy(
        dtype=int
    )
    early_choice = rng_for(61, case_index, 0).permutation(early_ids)[:8]

    remaining = np.setdiff1d(
        np.arange(len(paths)), early_choice, assume_unique=False
    )
    other_choice = rng_for(61, case_index, 1).choice(
        remaining,
        size=min(16, len(remaining)),
        replace=False,
    )
    selected_paths = np.concatenate([early_choice, other_choice])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for path_id in selected_paths:
        stop = int(events.loc[path_id, "stopping_step"])
        axes[0].plot(
            times[:stop + 1],
            paths[path_id, :stop + 1],
            color="tab:blue",
            alpha=0.25,
            linewidth=0.8,
        )
        if bool(events.loc[path_id, "paid_early"]):
            axes[0].scatter(
                times[stop],
                paths[path_id, stop],
                color="tab:red",
                edgecolor="white",
                linewidth=0.4,
                s=45,
                zorder=4,
            )

    axes[0].axhline(
        contract.strike, color="black", ls="--", label="Strike"
    )
    axes[0].scatter(
        [], [], color="tab:red", s=45, label="Positive early exercise"
    )
    axes[0].set(
        xlabel="Time (years)",
        ylabel="Stock price",
        title="Evaluation paths and their exercise decisions",
    )
    axes[0].legend(fontsize=8)

    early_events = events[events["paid_early"]]
    # Cap displayed points only; every event remains in the exported CSV.
    if len(early_events) > 12000:
        indices = rng_for(62, case_index).choice(
            len(early_events), 12000, replace=False
        )
        displayed_events = early_events.iloc[indices]
    else:
        displayed_events = early_events

    if len(displayed_events):
        scatter = axes[1].scatter(
            displayed_events["stopping_time"],
            displayed_events["stock_at_stopping"],
            c=displayed_events["cashflow"],
            cmap="viridis",
            s=9,
            alpha=0.5,
            edgecolors="none",
        )
        fig.colorbar(
            scatter, ax=axes[1], label="Exercise payoff"
        )
    else:
        axes[1].text(
            0.5, 0.5, "No positive-payoff early exercises",
            ha="center", va="center", transform=axes[1].transAxes
        )

    axes[1].axhline(contract.strike, color="black", ls="--")
    axes[1].set(
        xlabel="Exercise time (years)",
        ylabel="Stock price at exercise",
        title=f"Exercise events: {len(early_events):,} in full sample",
    )

    fig.suptitle(case_name.replace("_", " "), fontsize=13)
    finish_figure(fig, f"{case_name}_paths_and_exercise_events")

    # Heatmap: intermediate dates only; root decision is reported separately.
    spot_edges = np.linspace(
        0.4 * contract.strike, 1.6 * contract.strike, 49
    )
    alive_counts = np.zeros((len(spot_edges) - 1, steps - 1))
    exercise_counts = np.zeros_like(alive_counts)

    stopping = events["stopping_step"].to_numpy()
    cashflows = events["cashflow"].to_numpy()

    for step in range(1, steps):
        alive = stopping >= step
        exercised = (stopping == step) & (cashflows > 0)

        alive_counts[:, step - 1] = np.histogram(
            paths[alive, step], bins=spot_edges
        )[0]
        exercise_counts[:, step - 1] = np.histogram(
            paths[exercised, step], bins=spot_edges
        )[0]

    minimum_bin_count = 20
    frequency = np.divide(
        exercise_counts,
        alive_counts,
        out=np.full_like(alive_counts, np.nan),
        where=alive_counts >= minimum_bin_count,
    )

    fig, ax = plt.subplots(figsize=(11, 5))
    if np.isfinite(frequency).any():
        time_edges = (
            np.arange(steps) + 0.5
        ) * contract.maturity / steps

        mesh = ax.pcolormesh(
            time_edges,
            spot_edges,
            frequency,
            shading="flat",
            cmap="magma",
            norm=Normalize(0, 1),
        )
        fig.colorbar(
            mesh, ax=ax,
            label="Exercise frequency conditional on alive/bin"
        )
    else:
        ax.text(
            0.5, 0.5,
            "No populated intermediate-date bins under this policy",
            ha="center", va="center", transform=ax.transAxes
        )

    ax.axhline(contract.strike, color="cyan", ls="--", lw=1)
    ax.set(
        xlabel="Decision time (years)",
        ylabel="Stock-price bin",
        title=(
            case_name.replace("_", " ")
            + f": exercise frequency; minimum bin count {minimum_bin_count}"
        ),
    )
    finish_figure(fig, f"{case_name}_exercise_frequency")

    for bin_index in range(len(spot_edges) - 1):
        for column in range(steps - 1):
            exercise_heatmap_tables.append({
                "case": case_name,
                "time": (column + 1) * contract.maturity / steps,
                "spot_lower": spot_edges[bin_index],
                "spot_upper": spot_edges[bin_index + 1],
                "alive_count": int(alive_counts[bin_index, column]),
                "exercise_count": int(exercise_counts[bin_index, column]),
                "conditional_frequency": frequency[bin_index, column],
            })

save_table(
    pd.DataFrame(exercise_heatmap_tables),
    "american_exercise_heatmap_bins",
)

In [ ]:
# Tree refinement: matched exercise dates remain fixed.
tree_rows = []
base_steps = AMERICAN_CFG["exercise_steps"]

for case_name, contract in AMERICAN_CASES.items():
    for substeps in (2, 4, 8, 16):
        total_steps = base_steps * substeps
        tree_rows.append({
            "case": case_name,
            "tree_steps": total_steps,
            "exercise_dates": base_steps,
            "american_tree": crr_value(
                contract, total_steps, exercise_every=1
            ),
            "matched_bermudan_tree": crr_value(
                contract, total_steps, exercise_every=substeps
            ),
            "european_tree": crr_value(contract, total_steps),
            "european_bsm": bsm_price(contract),
        })

tree_refinement = save_table(
    pd.DataFrame(tree_rows), "american_tree_refinement"
)
display(tree_refinement)

# Nested exercise schedules, but independently refitted policies.
# Monte Carlo estimates are not forced to be monotone.
sensitivity_rows = []
degrees = (1, 2, 3)
exercise_grids = (12, 24, 48)

for case_index, (case_name, contract) in enumerate(AMERICAN_CASES.items()):
    for grid_index, steps in enumerate(exercise_grids):
        training_paths = exercise_paths(
            contract,
            AMERICAN_CFG["train_paths"],
            steps,
            rng_for(63, case_index, grid_index, 0),
        )
        evaluation_paths = exercise_paths(
            contract,
            AMERICAN_CFG["evaluation_paths"],
            steps,
            rng_for(63, case_index, grid_index, 1),
        )

        matched_tree = crr_value(
            contract,
            steps * AMERICAN_CFG["tree_substeps"],
            exercise_every=AMERICAN_CFG["tree_substeps"],
        )

        for degree in degrees:
            policy = fit_exercise_policy(
                contract,
                training_paths,
                degree=degree,
                minimum=AMERICAN_CFG["min_regression"],
            )
            result = evaluate_exercise_policy(
                contract, evaluation_paths, policy
            )

            sensitivity_rows.append({
                "case": case_name,
                "exercise_steps": steps,
                "degree": degree,
                "selected_policy": policy["selected_policy"],
                **result["summary"],
                "matched_tree": matched_tree,
                "difference_from_tree": (
                    result["summary"]["estimate"] - matched_tree
                ),
                "fitted_dates": len(policy["models"]),
            })

        del training_paths, evaluation_paths

american_sensitivity = save_table(
    pd.DataFrame(sensitivity_rows),
    "american_grid_and_basis_sensitivity",
)
display(american_sensitivity)

fig, axes = plt.subplots(
    1, len(AMERICAN_CASES), figsize=(15, 4.5)
)

for ax, case_name in zip(axes, AMERICAN_CASES):
    subset = american_sensitivity[
        american_sensitivity["case"] == case_name
    ]

    for degree, group in subset.groupby("degree"):
        group = group.sort_values("exercise_steps")
        ax.errorbar(
            group["exercise_steps"],
            group["estimate"],
            yerr=np.vstack([
                group["estimate"] - group["ci_low"],
                group["ci_high"] - group["estimate"],
            ]),
            fmt="o-",
            capsize=3,
            label=f"Polynomial degree {degree}",
        )

    reference = subset.drop_duplicates(
        "exercise_steps"
    ).sort_values("exercise_steps")

    ax.plot(
        reference["exercise_steps"],
        reference["matched_tree"],
        "k--",
        label="Matched-grid tree",
    )
    ax.set(
        xlabel="Exercise intervals",
        ylabel="Policy value",
        title=case_name.replace("_", " "),
    )
    ax.legend(fontsize=8)

finish_figure(fig, "american_grid_and_basis_sensitivity")

# Add to the existing final run manifest.
CFG["american"] = AMERICAN_CFG.copy()

### American-option review checklist

Before interpreting the charts:

1. Inspect the selected policy and regression diagnostics.
2. Check tree refinement; agreement with one tree resolution is not enough.
3. Examine differences across polynomial degree and exercise schedule.
4. Inspect no-dividend calls for suspicious early exercise.
5. Keep any negative estimated premium versus holding to maturity visible.
6. Distinguish the conditional Monte Carlo interval from total pricing error.
7. Repeat training and valuation with new seeds before making robustness claims.
8. Never relabel an observed exercise-event cloud as the optimal boundary.

The early-exercise premium estimate uses paired policy and maturity-only cash
flows on the same evaluation paths. Its uncertainty is therefore calculated
from the pathwise difference.

An increase in exercise opportunities increases the true optimal value for
nested schedules, but independently fitted finite-sample LSMC estimates need
not increase monotonically.

This module does not calculate a dual upper bound or quantify the complete
optimal-stopping error. It supplies a separately evaluated candidate policy,
tree comparisons, and sensitivity diagnostics.

Reference:
Longstaff and Schwartz (2001), “Valuing American Options by Simulation:
A Simple Least-Squares Approach.”


### **➤ BARRIER OPTIONS**



## Single-barrier options: knock-in, knock-out and monitoring

### Contracts

This section values European-style single-barrier calls and puts under the
existing risk-neutral GBM model.

| Feature | Convention |
|---|---|
| Barrier direction | Up or down |
| Activation | Knock-in or knock-out |
| Monitoring | Equally spaced discrete dates, including time zero and maturity |
| Up-barrier hit | Observed stock price greater than or equal to the barrier |
| Down-barrier hit | Observed stock price less than or equal to the barrier |
| Rebate | Zero |
| Payoff timing | Maturity |
| Vanilla payoff | Call or put payoff at maturity |
| Exercise before maturity | Not permitted |

A knock-in pays the vanilla payoff only if the barrier has been hit.
A knock-out pays the vanilla payoff only if the barrier has not been hit.

If the barrier is already breached at time zero, the knock-in is activated
immediately and the knock-out is extinguished immediately.

### Valuation

Let H denote the event that the barrier is hit at one of the monitoring dates,
and let g(S_T) be the vanilla terminal payoff.

The discounted pathwise payoffs are:

    Knock-in  = exp(-rT) × g(S_T) × 1(H)
    Knock-out = exp(-rT) × g(S_T) × 1(not H)

Consequently, on every simulated path:

    Knock-in + Knock-out = discounted vanilla payoff

This identity requires matching contract terms and zero rebates.

### Simulation and uncertainty

Stock prices are simulated using exact GBM transitions at the monitoring
dates. No antithetic sampling or moment matching is applied in this module.

Prices, standard errors and approximate 95% confidence intervals are calculated
from independent pathwise discounted payoffs.

Knock-in, knock-out and vanilla values share the same simulated paths.
Their estimates are dependent. Portfolio uncertainty and parity checks must
therefore use pathwise sums or differences, not sums of individual standard
errors.

The barrier-hit frequency is not the probability of receiving a positive
payoff: an activated option may still expire out of the money.

### Monitoring limitation

Exact GBM transitions do not detect crossings between observation dates.

This is intentional for a discretely monitored contract. It would be a
modelling error to describe these prices as continuously monitored barrier
prices.

The monitoring experiment uses nested schedules sampled from the same fine
paths. Increasing the number of observations can only add observed hits on
those paths. It therefore increases knock-in payoffs and decreases knock-out
payoffs pathwise. Each schedule nevertheless defines a different contract.

### Validation scope

The module checks:

- Pathwise in–out parity.
- Nonnegative payoffs.
- Knock-in and knock-out payoffs bounded by the vanilla payoff.
- Consistent treatment of an initially breached barrier.
- Agreement between stored terminal prices and the existing terminal simulator.
- Pathwise monitoring monotonicity for nested schedules.

The vanilla Monte Carlo estimate is compared with BSM. No independent
closed-form benchmark is supplied for the discrete barrier prices.

Passing parity checks does not establish complete pricing accuracy: the
same mistake could affect both complementary barrier payoffs.

In [ ]:
BARRIER_CFG = {
    "paths": 20000 if MODE == "quick" else 150000,
    "monitoring_steps": 96,
    "monitoring_grids": [12, 24, 48, 96],
    "spot_grid_points": 13,
    "display_paths": 24,
}

required_barrier_objects = (
    "BASE", "CFG", "Contract", "payoff", "terminal", "bsm_price",
    "summarize", "rng_for", "save_table", "finish_figure",
)
missing = [
    name for name in required_barrier_objects if name not in globals()
]
if missing:
    raise RuntimeError(
        "Run the earlier notebook cells first. Missing: "
        + ", ".join(missing)
    )


def barrier_paths(contract, n_paths, steps, rng):
    if contract.maturity <= 0:
        raise ValueError("Barrier experiments require positive maturity.")
    if not isinstance(n_paths, (int, np.integer)) or n_paths < 2:
        raise ValueError("n_paths must be an integer >= 2.")
    if not isinstance(steps, (int, np.integer)) or steps < 1:
        raise ValueError("steps must be a positive integer.")

    dt = contract.maturity / steps
    paths = np.empty((n_paths, steps + 1), dtype=float)
    paths[:, 0] = contract.spot

    drift = (
        contract.rate - contract.dividend - 0.5 * contract.vol**2
    ) * dt
    diffusion = contract.vol * np.sqrt(dt)

    for step in range(1, steps + 1):
        z = rng.standard_normal(n_paths)
        paths[:, step] = paths[:, step - 1] * np.exp(
            drift + diffusion * z
        )

    return paths


def barrier_payoffs(contract, paths, barrier, direction):
    if not np.isfinite(barrier) or barrier <= 0:
        raise ValueError("Barrier must be finite and positive.")
    if direction not in {"up", "down"}:
        raise ValueError("direction must be 'up' or 'down'.")
    if paths.ndim != 2 or paths.shape[1] < 2:
        raise ValueError("Expected a two-dimensional monitored path array.")
    if not np.all(np.isfinite(paths)) or np.any(paths <= 0):
        raise ValueError("Stock paths must be finite and positive.")
    if not np.allclose(paths[:, 0], contract.spot):
        raise ValueError("Initial path values must match contract.spot.")

    crossings = (
        paths >= barrier if direction == "up"
        else paths <= barrier
    )
    hit = crossings.any(axis=1)

    # -1 marks paths with no observed hit.
    first_hit = np.full(len(paths), -1, dtype=int)
    first_hit[hit] = np.argmax(crossings[hit], axis=1)

    vanilla = (
        np.exp(-contract.rate * contract.maturity)
        * payoff(paths[:, -1], contract)
    )
    knock_in = np.where(hit, vanilla, 0.0)
    knock_out = np.where(hit, 0.0, vanilla)

    return {
        "vanilla": vanilla,
        "knock_in": knock_in,
        "knock_out": knock_out,
        "hit": hit,
        "first_hit": first_hit,
    }


def barrier_summary_rows(contract, result, case, barrier, direction, steps):
    rows = []
    for product in ("vanilla", "knock_in", "knock_out"):
        row = summarize(result[product])
        row.update({
            "case": case,
            "kind": contract.kind,
            "direction": direction,
            "barrier": barrier,
            "monitoring_steps": steps,
            "product": product,
            "hit_fraction": float(result["hit"].mean()),
            "positive_payoff_fraction": float(
                np.mean(result[product] > 0)
            ),
        })

        # Only vanilla has an analytical benchmark in this module.
        row["benchmark"] = (
            bsm_price(contract) if product == "vanilla" else np.nan
        )
        row["benchmark_error"] = (
            row["estimate"] - row["benchmark"]
            if product == "vanilla" else np.nan
        )
        rows.append(row)
    return rows

In [ ]:
# Generate common stock paths once. Call/put payoff comparisons share scenarios.
barrier_base_paths = barrier_paths(
    BASE,
    BARRIER_CFG["paths"],
    BARRIER_CFG["monitoring_steps"],
    rng_for(70, 0),
)

BARRIER_CASES = {}
for kind in ("call", "put"):
    contract = replace(BASE, kind=kind)
    BARRIER_CASES[f"{kind}_down"] = (
        contract, "down", 0.85 * BASE.spot
    )
    BARRIER_CASES[f"{kind}_up"] = (
        contract, "up", 1.15 * BASE.spot
    )

barrier_checks = []
barrier_rows = []
barrier_event_frames = []


def barrier_check(name, condition):
    passed = bool(condition)
    barrier_checks.append({"check": name, "passed": passed})
    if not passed:
        raise AssertionError(name)


# Verify exact-transition terminal consistency using a small shared draw matrix.
test_steps = 8
test_z = rng_for(70, 1).standard_normal((1000, test_steps))
test_dt = BASE.maturity / test_steps
test_terminal = BASE.spot * np.exp(
    np.sum(
        (
            BASE.rate - BASE.dividend - 0.5 * BASE.vol**2
        ) * test_dt
        + BASE.vol * np.sqrt(test_dt) * test_z,
        axis=1,
    )
)
equivalent_z = test_z.sum(axis=1) / np.sqrt(test_steps)
barrier_check(
    "exact_transition_terminal_consistency",
    np.allclose(
        test_terminal, terminal(BASE, equivalent_z),
        rtol=1e-12, atol=1e-12
    ),
)

for case_index, (case_name, specification) in enumerate(
    BARRIER_CASES.items()
):
    contract, direction, barrier = specification
    steps = BARRIER_CFG["monitoring_steps"]
    result = barrier_payoffs(
        contract, barrier_base_paths, barrier, direction
    )

    barrier_check(
        f"{case_name}_pathwise_parity",
        np.allclose(
            result["knock_in"] + result["knock_out"],
            result["vanilla"],
            rtol=1e-12, atol=1e-12,
        ),
    )
    for product in ("knock_in", "knock_out"):
        barrier_check(
            f"{case_name}_{product}_bounds",
            np.all(result[product] >= 0)
            and np.all(result[product] <= result["vanilla"]),
        )

    # Equality at time zero counts as a hit in either direction.
    initially_hit = barrier_payoffs(
        contract, barrier_base_paths, contract.spot, direction
    )
    barrier_check(
        f"{case_name}_initial_touch",
        np.all(initially_hit["hit"])
        and np.all(initially_hit["first_hit"] == 0)
        and np.all(initially_hit["knock_out"] == 0)
        and np.array_equal(
            initially_hit["knock_in"], initially_hit["vanilla"]
        ),
    )

    barrier_rows.extend(
        barrier_summary_rows(
            contract, result, case_name, barrier, direction, steps
        )
    )

    hit_ids = np.flatnonzero(result["hit"])
    event_frame = pd.DataFrame({
        "case": case_name,
        "path_id": hit_ids,
        "first_hit_step": result["first_hit"][hit_ids],
        "first_hit_time": (
            result["first_hit"][hit_ids] * contract.maturity / steps
        ),
        "stock_at_first_hit": barrier_base_paths[
            hit_ids, result["first_hit"][hit_ids]
        ],
        "terminal_stock": barrier_base_paths[hit_ids, -1],
        "discounted_vanilla_payoff": result["vanilla"][hit_ids],
    })
    barrier_event_frames.append(event_frame)

    # Stratification is for illustration only, not probability estimation.
    miss_ids = np.flatnonzero(~result["hit"])
    plot_rng = rng_for(71, case_index)
    selected = np.concatenate([
        plot_rng.permutation(hit_ids)[:12],
        plot_rng.permutation(miss_ids)[:12],
    ])

    times = np.linspace(0, contract.maturity, steps + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for path_id in selected:
        hit = result["hit"][path_id]
        axes[0].plot(
            times,
            barrier_base_paths[path_id],
            color="tab:red" if hit else "tab:blue",
            alpha=0.35,
            linewidth=0.8,
        )
        if hit:
            step = result["first_hit"][path_id]
            axes[0].scatter(
                times[step],
                barrier_base_paths[path_id, step],
                color="black", s=25, zorder=4,
            )

    axes[0].axhline(
        barrier, color="tab:orange", ls="--", label="Barrier"
    )
    axes[0].axhline(
        contract.strike, color="grey", ls=":", label="Strike"
    )
    axes[0].plot([], [], color="tab:red", label="Observed hit")
    axes[0].plot([], [], color="tab:blue", label="No observed hit")
    axes[0].scatter([], [], color="black", s=25, label="First observed hit")
    axes[0].set(
        xlabel="Time (years)",
        ylabel="Stock price",
        title="Illustrative paths—not a representative hit-rate sample",
    )
    axes[0].legend(fontsize=8)

    displayed = event_frame
    if len(displayed) > 10000:
        displayed = displayed.iloc[
            plot_rng.choice(len(displayed), 10000, replace=False)
        ]

    if len(displayed):
        scatter = axes[1].scatter(
            displayed["first_hit_time"],
            displayed["stock_at_first_hit"],
            c=displayed["terminal_stock"],
            cmap="viridis", s=8, alpha=0.45,
            edgecolors="none",
        )
        fig.colorbar(
            scatter, ax=axes[1], label="Terminal stock price"
        )
    else:
        axes[1].text(
            0.5, 0.5, "No observed hits",
            transform=axes[1].transAxes,
            ha="center", va="center",
        )

    axes[1].axhline(barrier, color="tab:orange", ls="--")
    axes[1].set(
        xlabel="First observed hit time (years)",
        ylabel="Stock price at first observed hit",
        title="Discrete monitoring: overshoot is possible",
    )
    fig.suptitle(case_name.replace("_", " "))
    finish_figure(fig, f"barrier_{case_name}_paths_and_hits")

barrier_prices = save_table(
    pd.DataFrame(barrier_rows), "barrier_price_comparison"
)
save_table(
    pd.concat(barrier_event_frames, ignore_index=True),
    "barrier_first_hit_events",
)
display(barrier_prices)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, case_name in zip(axes.ravel(), BARRIER_CASES):
    subset = barrier_prices[
        barrier_prices["case"] == case_name
    ].set_index("product").loc[["vanilla", "knock_in", "knock_out"]]

    ax.errorbar(
        np.arange(3),
        subset["estimate"],
        yerr=np.vstack([
            subset["estimate"] - subset["ci_low"],
            subset["ci_high"] - subset["estimate"],
        ]),
        fmt="o", capsize=5, color="tab:blue",
    )
    contract = BARRIER_CASES[case_name][0]
    ax.axhline(
        bsm_price(contract),
        color="black", ls="--", label="Vanilla BSM only"
    )
    ax.set_xticks(
        np.arange(3), ["Vanilla", "Knock-in", "Knock-out"]
    )
    ax.set(
        ylabel="Present value",
        title=case_name.replace("_", " "),
    )
    ax.legend(fontsize=8)

finish_figure(fig, "barrier_price_intervals")

### Barrier-level and monitoring experiments

The barrier-level experiment changes only the barrier while keeping the
simulated paths, vanilla payoff and monitoring schedule fixed. Estimates
at different barrier levels are therefore correlated.

For an up barrier, moving the barrier farther above spot reduces the hit set.
For a down barrier, moving it farther below spot reduces the hit set.

The monitoring experiment uses the nested schedules 12, 24, 48 and 96
intervals from the same 96-interval paths. Time zero and maturity remain
included in every schedule.

The plots show changes in contract values, not convergence to an independently
known continuous-barrier price. Pointwise sampling intervals exclude model
risk and do not form simultaneous confidence bands.

In [ ]:
barrier_level_rows = []
monitoring_rows = []

for case_name, (contract, direction, default_barrier) in BARRIER_CASES.items():
    ratios = (
        np.linspace(1.02, 1.40, BARRIER_CFG["spot_grid_points"])
        if direction == "up"
        else np.linspace(0.60, 0.98, BARRIER_CFG["spot_grid_points"])
    )

    previous = None
    for ratio in ratios:
        barrier = contract.spot * ratio
        result = barrier_payoffs(
            contract, barrier_base_paths, barrier, direction
        )

        barrier_level_rows.extend(
            barrier_summary_rows(
                contract, result, case_name, barrier, direction,
                BARRIER_CFG["monitoring_steps"],
            )
        )

        if previous is not None:
            # Ratios increase in both directions.
            # Raising an up barrier reduces hits.
            # Raising a down barrier increases hits.
            if direction == "up":
                ordered = np.all(
                    result["knock_in"] <= previous["knock_in"]
                )
            else:
                ordered = np.all(
                    result["knock_in"] >= previous["knock_in"]
                )

            barrier_check(
                f"{case_name}_barrier_order_{ratio:.6f}", ordered
            )
        previous = result

    previous = None
    fine_steps = BARRIER_CFG["monitoring_steps"]

    for monitoring_steps in BARRIER_CFG["monitoring_grids"]:
        if fine_steps % monitoring_steps != 0:
            raise ValueError(
                "Every monitoring grid must divide the fine grid."
            )

        stride = fine_steps // monitoring_steps
        observed_paths = barrier_base_paths[:, ::stride]
        result = barrier_payoffs(
            contract, observed_paths, default_barrier, direction
        )

        monitoring_rows.extend(
            barrier_summary_rows(
                contract, result, case_name, default_barrier,
                direction, monitoring_steps,
            )
        )

        if previous is not None:
            barrier_check(
                f"{case_name}_nested_monitoring_{monitoring_steps}",
                np.all(result["knock_in"] >= previous["knock_in"])
                and np.all(
                    result["knock_out"] <= previous["knock_out"]
                ),
            )
        previous = result

barrier_levels = save_table(
    pd.DataFrame(barrier_level_rows), "barrier_level_sensitivity"
)
barrier_monitoring = save_table(
    pd.DataFrame(monitoring_rows), "barrier_monitoring_schedules"
)
save_table(
    pd.DataFrame(barrier_checks), "barrier_validation_checks"
)

for table, x_column, figure_name, x_label in (
    (
        barrier_levels,
        "barrier",
        "barrier_level_sensitivity",
        "Barrier level",
    ),
    (
        barrier_monitoring,
        "monitoring_steps",
        "barrier_monitoring_sensitivity",
        "Monitoring intervals",
    ),
):
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    for ax, case_name in zip(axes.ravel(), BARRIER_CASES):
        subset = table[table["case"] == case_name]

        for product, colour in (
            ("knock_in", "tab:orange"),
            ("knock_out", "tab:blue"),
        ):
            group = subset[
                subset["product"] == product
            ].sort_values(x_column)

            ax.errorbar(
                group[x_column],
                group["estimate"],
                yerr=np.vstack([
                    group["estimate"] - group["ci_low"],
                    group["ci_high"] - group["estimate"],
                ]),
                fmt="o-",
                markersize=3,
                capsize=2,
                color=colour,
                label=product.replace("_", " "),
            )

        contract = BARRIER_CASES[case_name][0]
        ax.axhline(
            bsm_price(contract),
            color="black", ls="--", label="Vanilla BSM"
        )
        ax.set(
            xlabel=x_label,
            ylabel="Present value",
            title=case_name.replace("_", " "),
        )
        ax.legend(fontsize=8)

    finish_figure(fig, figure_name)

CFG["barrier"] = BARRIER_CFG.copy()

display(
    pd.DataFrame([{
        "barrier_checks": len(barrier_checks),
        "all_passed": all(row["passed"] for row in barrier_checks),
        "monitoring": "Discrete; includes time zero and maturity",
        "rebate": 0.0,
        "payoff_time": "Maturity",
    }])
)

# Large path arrays are no longer required after the exports and plots.
del barrier_base_paths

### Barrier results: interpretation and limitations

Review the following before writing the final study conclusions:

1. Confirm every deterministic barrier check passed.
2. Compare the vanilla Monte Carlo estimate with BSM.
3. Check hit fractions and positive-payoff fractions separately.
4. Inspect barrier-level sensitivity on the correct side of spot.
5. Verify nested-monitoring behaviour on shared paths.
6. Do not interpret zero sampled knock-in or knock-out payoffs as proof of
   a zero contract value.
7. Remember that confidence intervals describe sampling uncertainty, not
   complete pricing accuracy.
8. Do not describe first observed crossings as continuous first-passage times.

This module does not implement continuous monitoring, bridge corrections,
rebates, moving barriers, double barriers, American barrier exercise,
or barrier Greeks.

It completes the planned product scope for this notebook version.
The next stage is execution, numerical review, debugging, and documentation,
rather than adding further option types.


### **➤ SAVING RESULTS**



In [19]:
# FINAL CELL: run AFTER all analysis and the existing manifest/checksum cell.

from pathlib import Path
from datetime import datetime, timezone
from dataclasses import is_dataclass, asdict
import csv
import hashlib
import inspect
import json
import platform
import shutil
import sys
import zipfile

from IPython.display import display, FileLink


# OPTIONAL OVERRIDE:
# If automatic discovery fails, enter the actual notebook output folders.
# Example: EXPORT_SOURCE_DIRS = ["outputs/figures", "outputs/tables"]
EXPORT_SOURCE_DIRS = []

# Optional: include a SAVED copy of this notebook by entering its exact path.
# Save the notebook first. Review it for secrets before including it.
NOTEBOOK_TO_INCLUDE = None

# Only recognised result formats are collected automatically.
RESULT_EXTENSIONS = {
    ".csv", ".tsv", ".png", ".jpg", ".jpeg",
    ".svg", ".pdf", ".json", ".parquet", ".npz",
}

# Packaging destinations are excluded from collection on later runs.
PACKAGE_PARENT = Path("results_packages").resolve()
PACKAGE_PARENT.mkdir(parents=True, exist_ok=True)

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
package_name = f"option_pricing_results_{run_stamp}"
package_dir = PACKAGE_PARENT / package_name
zip_path = PACKAGE_PARENT / f"{package_name}.zip"


def _inside(path, parent):
    try:
        path.relative_to(parent)
        return True
    except ValueError:
        return False


def _json_safe(value):
    if is_dataclass(value) and not isinstance(value, type):
        return _json_safe(asdict(value))
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if hasattr(value, "tolist"):
        return _json_safe(value.tolist())
    return repr(value)


def _write_json(path, data):
    path.write_text(
        json.dumps(_json_safe(data), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def _sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# Discover directories directly referenced by the existing export helpers.
candidate_dirs = list(EXPORT_SOURCE_DIRS)

if not candidate_dirs:
    for helper_name in ("save_table", "finish_figure"):
        helper = globals().get(helper_name)
        if not callable(helper):
            continue

        try:
            closure = inspect.getclosurevars(helper)
            referenced = {**closure.globals, **closure.nonlocals}
        except (TypeError, ValueError):
            referenced = {}

        for name, value in referenced.items():
            label = name.lower()
            looks_like_output = any(
                word in label
                for word in ("out", "export", "figure", "table", "plot", "result")
            )
            if looks_like_output and isinstance(value, (str, Path)):
                path = Path(value).expanduser()
                if path.is_dir():
                    candidate_dirs.append(path)

    # Common output-directory globals, if the helpers delegate elsewhere.
    for name in (
        "OUT_DIR", "OUTPUT_DIR", "OUTPUT_ROOT", "OUT",
        "FIG_DIR", "FIGURE_DIR", "PLOT_DIR",
        "TABLE_DIR", "TABLES_DIR", "RESULTS_DIR", "EXPORT_DIR",
    ):
        value = globals().get(name)
        if isinstance(value, (str, Path)):
            path = Path(value).expanduser()
            if path.is_dir():
                candidate_dirs.append(path)

source_dirs = sorted({
    Path(path).expanduser().resolve()
    for path in candidate_dirs
    if Path(path).expanduser().is_dir()
}, key=str)

# Never automatically archive the working directory, home directory or repo.
unsafe_roots = {Path.cwd().resolve(), Path.home().resolve(), Path("/")}
for root in source_dirs:
    if root in unsafe_roots or (root / ".git").exists():
        raise RuntimeError(
            f"Refusing broad collection from {root}. "
            "Set EXPORT_SOURCE_DIRS to dedicated results subfolders."
        )

# Remove nested duplicates when a parent directory already covers them.
source_dirs = [
    root for root in source_dirs
    if not any(
        other != root and _inside(root, other)
        for other in source_dirs
    )
]

if not source_dirs:
    raise RuntimeError(
        "Could not identify your saved-results directories. "
        "Set EXPORT_SOURCE_DIRS at the top of this cell to the folders "
        "used by save_table and finish_figure, then rerun."
    )

# Collect only ordinary, non-hidden, non-symlink result files.
files_to_copy = []

for source_index, root in enumerate(source_dirs, start=1):
    if _inside(root, PACKAGE_PARENT):
        continue

    for source in sorted(root.rglob("*")):
        relative = source.relative_to(root)

        if (
            source.is_symlink()
            or not source.is_file()
            or any(part.startswith(".") for part in relative.parts)
            or _inside(source.resolve(), PACKAGE_PARENT)
            or not _inside(source.resolve(), root)
            or source.suffix.lower() not in RESULT_EXTENSIONS
        ):
            continue

        destination = (
            Path("artifacts")
            / f"source_{source_index:02d}_{root.name}"
            / relative
        )
        files_to_copy.append((source, destination))

if not files_to_copy:
    raise RuntimeError(
        "The selected folders contain no recognised saved results. "
        "Run the analysis/export cells first or correct EXPORT_SOURCE_DIRS."
    )

# Check an optional notebook before creating the package.
notebook_path = None
if NOTEBOOK_TO_INCLUDE is not None:
    notebook_path = Path(NOTEBOOK_TO_INCLUDE).expanduser().resolve()
    if not notebook_path.is_file() or notebook_path.suffix.lower() != ".ipynb":
        raise ValueError(
            "NOTEBOOK_TO_INCLUDE must point to an existing .ipynb file."
        )

package_dir.mkdir(parents=True, exist_ok=False)

for source, relative_destination in files_to_copy:
    destination = package_dir / relative_destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

if notebook_path is not None:
    notebook_destination = package_dir / "notebook" / notebook_path.name
    notebook_destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(notebook_path, notebook_destination)

settings = {
    name: _json_safe(globals()[name])
    for name in ("MODE", "BASE", "CFG", "BARRIER_CFG")
    if name in globals()
}

_write_json(package_dir / "configuration_snapshot.json", settings)

versions = {}
for module_name in ("numpy", "pandas", "scipy", "matplotlib"):
    module = sys.modules.get(module_name)
    if module is not None:
        versions[module_name] = getattr(module, "__version__", "unknown")

_write_json(
    package_dir / "environment.json",
    {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "package_versions": versions,
        "source_directories": [str(root) for root in source_dirs],
        "copied_artifact_count": len(files_to_copy),
        "saved_notebook_included": notebook_path is not None,
    },
)

readme = """# Option-pricing results bundle

## Contents
- artifacts/: saved outputs copied from the notebook's export directories.
- configuration_snapshot.json: available settings captured at packaging time.
- environment.json: available environment information and source directories.
- inventory.csv: packaged file sizes and SHA-256 checksums.
- SHA256SUMS.txt: checksums for all other packaged files.
- notebook/: a saved notebook, only if explicitly supplied.

## Important limitations
Packaging does not rerun analysis or certify financial or numerical accuracy.
Files already present in the source folders may come from earlier runs.
A configuration snapshot does not prove every artifact used those settings.
Previously exported validation tables are preserved, not reinterpreted.
Unsaved notebook state, live figures, and unexported tables are not captured.
The notebook file alone does not guarantee full reproducibility.

Review validation tables, benchmark comparisons, filenames and timestamps
before treating this bundle as a completed study or publishing it.
Review all contents for sensitive information before sharing.
"""
(package_dir / "README.md").write_text(readme, encoding="utf-8")

# Inventory excludes itself and SHA256SUMS.txt to avoid circular checksums.
inventory_rows = []
for path in sorted(package_dir.rglob("*")):
    if path.is_file():
        inventory_rows.append({
            "relative_path": path.relative_to(package_dir).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": _sha256(path),
        })

with (package_dir / "inventory.csv").open(
    "w", newline="", encoding="utf-8"
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(inventory_rows)

# Includes inventory.csv, but necessarily excludes the checksum file itself.
checksum_lines = [
    f"{_sha256(path)}  {path.relative_to(package_dir).as_posix()}"
    for path in sorted(package_dir.rglob("*"))
    if path.is_file()
]
(package_dir / "SHA256SUMS.txt").write_text(
    "\n".join(checksum_lines) + "\n", encoding="utf-8"
)

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
) as archive:
    for path in sorted(package_dir.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                arcname=(
                    Path(package_name) / path.relative_to(package_dir)
                ).as_posix(),
            )

with zipfile.ZipFile(zip_path) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP integrity check failed: {bad_member}")
    archived_files = len(archive.namelist())

if not zip_path.is_file() or zip_path.stat().st_size == 0:
    raise RuntimeError("ZIP creation failed.")

print(f"Results folder: {package_dir}")
print(f"Archive: {zip_path.name}")
print(f"Packaged files: {archived_files}")
print(f"ZIP size: {zip_path.stat().st_size / (1024 ** 2):.2f} MiB")
print(f"ZIP SHA-256: {_sha256(zip_path)}")
print("ZIP integrity check: PASSED")
print("Analysis correctness: NOT assessed by this packaging cell")

# Google Colab download; otherwise show a local Jupyter download link.
try:
    from google.colab import files as colab_files
except ImportError:
    relative_zip = zip_path.relative_to(Path.cwd().resolve())
    display(FileLink(
        relative_zip.as_posix(),
        result_html_prefix="Download results ZIP: ",
    ))
else:
    colab_files.download(str(zip_path))

Results folder: /content/results_packages/option_pricing_results_20261117T051457_630200Z
Archive: option_pricing_results_20261117T051457_630200Z.zip
Packaged files: 56
ZIP size: 8.90 MiB
ZIP SHA-256: f31255acb70cbe8bc98a25a348cd28a9640bd02241b61333740d37dda14ac4bb
ZIP integrity check: PASSED
Analysis correctness: NOT assessed by this packaging cell


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>